# Men

In [1]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier # SỬ DỤNG CLASSIFIER CHO CONVERSION
from sklift.models import TwoModels
import warnings

# Bỏ qua cảnh báo UserWarning phiền phức từ sklearn/lightgbm
warnings.filterwarnings("ignore", category=UserWarning)

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data for T-Learner Conversion...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion' # BÀI TOÁN CONVERSION
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test
# Chú ý: .values để xóa Pandas format, y ép kiểu .astype(int) vì là nhãn nhị phân
X_train = train_df[in_features].values
y_train = train_df[label_feature].values.astype(int)
t_train = train_df[treatment_feature].values.astype(int)

X_val = val_df[in_features].values
y_val_true = val_df[label_feature].values.flatten().astype(int)
t_val_true = val_df[treatment_feature].values.flatten().astype(int)

X_test = test_df[in_features].values
y_test_true = test_df[label_feature].values.flatten().astype(int)
t_test_true = test_df[treatment_feature].values.flatten().astype(int)

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH 5 SEEDS)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        # Sử dụng LGBMClassifier cho Treatment và Control
        model_trmnt = LGBMClassifier(**params, random_state=SEED, verbose=-1)
        model_ctrl = LGBMClassifier(**params, random_state=SEED, verbose=-1)
        
        # Bọc vào TwoModels (T-Learner)
        t_learner = TwoModels(estimator_trmnt=model_trmnt, estimator_ctrl=model_ctrl, method='vanilla')
        
        # Huấn luyện mô hình
        t_learner.fit(X=X_train, y=y_train, treatment=t_train)
        
        # Dự đoán Uplift trên tập Val (TwoModels tự động tính P(Y=1|T=1) - P(Y=1|T=0))
        uplift_pred_val = t_learner.predict(X_val)
        
        # Đánh giá bằng AUQC
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (T-Learner Robust Conversion)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="T_Learner_Robust_Conv", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")

print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả list seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ T-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)")
print("="*50)

test_results = []

for SEED in seeds:
    # Set seed cho best params
    final_model_trmnt = LGBMClassifier(**best_params, random_state=SEED, verbose=-1)
    final_model_ctrl = LGBMClassifier(**best_params, random_state=SEED, verbose=-1)
    
    final_t_learner = TwoModels(estimator_trmnt=final_model_trmnt, estimator_ctrl=final_model_ctrl, method='vanilla')
    
    # Train
    final_t_learner.fit(X=X_train, y=y_train, treatment=t_train)
    
    # Predict
    uplift_pred_test = final_t_learner.predict(X_test)
    
    # Đánh giá Metrics
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán kết quả trung bình
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH T-LEARNER CONVERSION ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu kết quả
csv_filename = "t_learner_conversion_robust_results_men.csv"
results_df.to_csv(csv_filename, index=False)
print(f"\n💾 Đã lưu kết quả chi tiết từng seed vào: {csv_filename}")

/home/datnghiemxuan/hai/.venv_3_12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data for T-Learner Conversion...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (T-Learner Robust Conversion)...


Best trial: 31. Best value: 0.681849: 100%|██████████| 50/50 [35:38<00:00, 42.77s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.6818
Bộ tham số Robust tốt nhất:
   n_estimators: 373
   learning_rate: 0.006025191173331987
   max_depth: 12
   num_leaves: 137
   min_child_samples: 166
   subsample: 0.9672039143473169
   colsample_bytree: 0.537742919284369

🚀 ĐÁNH GIÁ T-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)
Seed 412312: AUUC=0.581, AUQC=0.583, Lift=0.009, KRCC=0.073
Seed 42: AUUC=0.585, AUQC=0.587, Lift=0.008, KRCC=0.118
Seed 1874: AUUC=0.578, AUQC=0.580, Lift=0.009, KRCC=0.099
Seed 902745: AUUC=0.581, AUQC=0.582, Lift=0.009, KRCC=0.089
Seed 1: AUUC=0.579, AUQC=0.580, Lift=0.010, KRCC=0.068

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH T-LEARNER CONVERSION (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.581 ± 0.003
Mean AUQC: 0.582 ± 0.003
Mean Lift: 0.009 ± 0.001
Mean KRCC: 0.089 ± 0.020

💾 Đã lưu kết quả chi tiết từng seed vào: t_learner_conversion_robust_results_men.csv


# Women

In [2]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier # SỬ DỤNG CLASSIFIER CHO CONVERSION
from sklift.models import TwoModels
import warnings

# Bỏ qua cảnh báo UserWarning phiền phức từ sklearn/lightgbm
warnings.filterwarnings("ignore", category=UserWarning)

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data for T-Learner Conversion...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Women/train_women.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Women/test_women.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Women/val_women.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion' # BÀI TOÁN CONVERSION
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test
# Chú ý: .values để xóa Pandas format, y ép kiểu .astype(int) vì là nhãn nhị phân
X_train = train_df[in_features].values
y_train = train_df[label_feature].values.astype(int)
t_train = train_df[treatment_feature].values.astype(int)

X_val = val_df[in_features].values
y_val_true = val_df[label_feature].values.flatten().astype(int)
t_val_true = val_df[treatment_feature].values.flatten().astype(int)

X_test = test_df[in_features].values
y_test_true = test_df[label_feature].values.flatten().astype(int)
t_test_true = test_df[treatment_feature].values.flatten().astype(int)

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH 5 SEEDS)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        # Sử dụng LGBMClassifier cho Treatment và Control
        model_trmnt = LGBMClassifier(**params, random_state=SEED, verbose=-1)
        model_ctrl = LGBMClassifier(**params, random_state=SEED, verbose=-1)
        
        # Bọc vào TwoModels (T-Learner)
        t_learner = TwoModels(estimator_trmnt=model_trmnt, estimator_ctrl=model_ctrl, method='vanilla')
        
        # Huấn luyện mô hình
        t_learner.fit(X=X_train, y=y_train, treatment=t_train)
        
        # Dự đoán Uplift trên tập Val (TwoModels tự động tính P(Y=1|T=1) - P(Y=1|T=0))
        uplift_pred_val = t_learner.predict(X_val)
        
        # Đánh giá bằng AUQC
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (T-Learner Robust Conversion)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="T_Learner_Robust_Conv", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")

print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả list seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ T-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)")
print("="*50)

test_results = []

for SEED in seeds:
    # Set seed cho best params
    final_model_trmnt = LGBMClassifier(**best_params, random_state=SEED, verbose=-1)
    final_model_ctrl = LGBMClassifier(**best_params, random_state=SEED, verbose=-1)
    
    final_t_learner = TwoModels(estimator_trmnt=final_model_trmnt, estimator_ctrl=final_model_ctrl, method='vanilla')
    
    # Train
    final_t_learner.fit(X=X_train, y=y_train, treatment=t_train)
    
    # Predict
    uplift_pred_test = final_t_learner.predict(X_test)
    
    # Đánh giá Metrics
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán kết quả trung bình
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH T-LEARNER CONVERSION ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu kết quả
csv_filename = "t_learner_conversion_robust_results_women.csv"
results_df.to_csv(csv_filename, index=False)
print(f"\n💾 Đã lưu kết quả chi tiết từng seed vào: {csv_filename}")

Loading data for T-Learner Conversion...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (T-Learner Robust Conversion)...


Best trial: 21. Best value: 0.823092: 100%|██████████| 50/50 [06:58<00:00,  8.37s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.8231
Bộ tham số Robust tốt nhất:
   n_estimators: 52
   learning_rate: 0.0011116853380816472
   max_depth: 3
   num_leaves: 145
   min_child_samples: 196
   subsample: 0.5060068016421219
   colsample_bytree: 0.5100079486661607

🚀 ĐÁNH GIÁ T-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)
Seed 412312: AUUC=0.562, AUQC=0.562, Lift=0.004, KRCC=0.005
Seed 42: AUUC=0.568, AUQC=0.568, Lift=0.005, KRCC=0.007
Seed 1874: AUUC=0.569, AUQC=0.569, Lift=0.004, KRCC=0.006
Seed 902745: AUUC=0.530, AUQC=0.530, Lift=0.004, KRCC=0.009
Seed 1: AUUC=0.561, AUQC=0.562, Lift=0.003, KRCC=0.042

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH T-LEARNER CONVERSION (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.558 ± 0.016
Mean AUQC: 0.558 ± 0.016
Mean Lift: 0.004 ± 0.001
Mean KRCC: 0.014 ± 0.016

💾 Đã lưu kết quả chi tiết từng seed vào: t_learner_conversion_robust_results_women.csv
